In [ ]:
# 1. Import thư viện
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
import seaborn as sns

In [ ]:
# 2. Load và tiền xử lý dữ liệu
df = pd.read_csv("application_train.csv")
df = df[['TARGET', 'DAYS_BIRTH', 'AMT_CREDIT', 'EXT_SOURCE_2', 'AMT_INCOME_TOTAL']].dropna()
df['DAYS_BIRTH'] = abs(df['DAYS_BIRTH'])  # lấy tuổi dương tính


In [ ]:
# 3. Chia tập train/test
X = df.drop(columns='TARGET')
y = df['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [ ]:
# 4. Huấn luyện mô hình LightGBM
model = LGBMClassifier(n_estimators=500, learning_rate=0.03, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)


In [ ]:
# 5. Dự đoán và đánh giá
y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_prob >= 0.5).astype(int)
auc_score = roc_auc_score(y_test, y_pred_prob)
f1 = f1_score(y_test, y_pred)
print("AUC:", auc_score)
print("F1-score:", f1)


In [ ]:
# 6. Quy đổi xác suất thành điểm tín dụng
score = 850 - 400 * np.log(y_pred_prob / (1 - y_pred_prob))  # công thức tương tự FICO
score = np.clip(score, 300, 850)


In [ ]:
# 7. Trực quan hóa kết quả
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# 7.1 Phân phối điểm tín dụng
sns.histplot(score, bins=40, kde=True, ax=axs[0])
axs[0].set_title('Phân phối điểm tín dụng')
axs[0].set_xlabel('Score')
axs[0].set_ylabel('Số lượng khách hàng')

# 7.2 ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axs[1].plot(fpr, tpr, label=f'AUC = {auc_score:.3f}')
axs[1].plot([0, 1], [0, 1], 'k--')
axs[1].set_title('ROC Curve')
axs[1].set_xlabel('False Positive Rate')
axs[1].set_ylabel('True Positive Rate')
axs[1].legend(loc='lower right')

# 7.3 Expected Loss theo nhóm rủi ro
df_eval = pd.DataFrame({'score': score, 'PD': y_pred_prob, 'EAD': X_test['AMT_CREDIT'], 'TARGET': y_test})
df_eval['LGD'] = 0.5
df_eval['EL'] = df_eval['PD'] * df_eval['LGD'] * df_eval['EAD']
df_eval['risk_group'] = pd.cut(df_eval['score'], bins=[0, 650, 750, 900], labels=['High', 'Medium', 'Low'])
el_by_group = df_eval.groupby('risk_group')['EL'].mean()

sns.barplot(x=el_by_group.index, y=el_by_group.values, ax=axs[2])
axs[2].set_title('Expected Loss theo nhóm rủi ro')
axs[2].set_ylabel('Expected Loss trung bình (VND)')
axs[2].set_xlabel('Nhóm rủi ro')

plt.tight_layout()
plt.show()
